## Phase I: Data Preparation & Cleaning
This notebook performs data preparation and cleaning for CO₂ emissions analysis using two datasets: `cars_2025.csv` (Malaysia 2025 car registrations) and `FuelConsumption.csv` (2014 fuel consumption and emission ratings).

### Step 1: Load the Datasets
We load the datasets using `pandas.read_csv()` to enable structured data analysis.

### Step 2: Normalize Join Keys
To ensure a successful merge between datasets, we normalize the `maker` and `model` fields by stripping whitespace and converting all text to lowercase. This helps avoid mismatches caused by inconsistent formatting.

### Step 3: Merge Registrations with Emissions Data
We merge both datasets using the normalized `maker` and `model` as keys. A left join (`how='left'`) ensures we retain all registration records, even if no emissions data is found.

### Step 4: Handle Missing CO₂ Values
We identify rows in the merged data where CO₂ emissions are missing. These are likely cars for which no match was found in the emissions dataset. We isolate these for inspection, and then drop them to create a clean dataset.

### Step 5: Remove Duplicates
We ensure the cleaned dataset does not contain exact duplicate rows using `drop_duplicates()`. This prevents bias or redundancy in later analysis.

### Step 6: Aggregations & Summaries
We compute key insights to understand emissions patterns:
- **6a:** Count total, matched, and unmatched records.
- **6b:** Average CO₂ emissions grouped by fuel type.
- **6c:** Average CO₂ emissions grouped by state.

### Step 7: Hierarchical Indexing and Pivot Table
We group emissions data by both `state` and `fuel`, creating a hierarchical index. This enables:
- Access to emissions data for a specific state.
- Construction of a pivot table with states as rows and fuel types as columns.

### Step 8: Combine State-Level and Fuel-Level Summaries
We consolidate the summary statistics from Step 6 into a single table using `pd.concat()`, making it easier to compare average emissions across different dimensions.

## STUDENT INFO: 
### 1) Name: 
##### - Muhammad Syahmi Faris bin Rusli
##### - Ahmad Adib Zikri bin A.Mazlam
##### - Muhammad Mukhritz Al Iman bin Mohd Raffi

### 2) Matric No:
##### - A23CS0138
##### - A23CS0205
##### - A23CS0250

### 3) Section: 03

In [2]:
import pandas as pd

1. **Loading two datasets into panda Dataframes**

In [4]:
# Phase I: Data Preparation & Cleaning

# 1. Load datasets
cars = pd.read_csv('cars_2025.csv')
fuel = pd.read_csv('FuelConsumption.csv')

2. **Creating new columns with cleaned and standardized versions of maker and model names**

In [6]:
# 2. Normalize join keys
cars['maker_norm']  = cars['maker'].str.strip().str.lower()
cars['model_norm']  = cars['model'].str.strip().str.lower()
fuel['maker_norm']  = fuel['MAKE'].str.strip().str.lower()
fuel['model_norm']  = fuel['MODEL'].str.strip().str.lower()

3. **Join cars with fuel using the normalized maker and model fields.**

In [8]:
# 3. Merge registrations with emissions data
merged = cars.merge(
    fuel,
    how='left',
    on=['maker_norm', 'model_norm']
)

4. **Filter out rows where no CO₂ emissions data was found**
   
   **Create a new DataFrame that only contains rows with valid CO₂ data**

In [10]:
# 4. Handle missing CO₂ values
#   a) isolate unmatched for manual review
unmatched = merged[ merged['CO2EMISSIONS'].isna() ]

#   b) drop rows with missing CO₂ for clean analysis
merged_clean = merged.dropna(subset=['CO2EMISSIONS']).copy()

5. **Remove any completely identical rows in the clean dataset**

In [12]:
# 5. Optional: check for and drop exact duplicate rows
merged_clean.drop_duplicates(inplace=True)

6. **Calculating total registration, matched and didn't match with emissions data C
7. ompute the average CO₂ emissions for each fuel type and average emissions by state**

In [14]:
# 6. Aggregations & summaries

# 6a. Counts summary
total_regs     = len(cars)
matched_regs   = len(merged_clean)
unmatched_regs = len(unmatched)

print("Total registrations:   ", total_regs)
print("Matched records:       ", matched_regs)
print("Unmatched records:     ", unmatched_regs)

# 6b. Average CO₂ emissions by fuel type
avg_by_fuel = (
    merged_clean
    .groupby('fuel')['CO2EMISSIONS']
    .mean()
    .sort_values()
    .reset_index()
)
print("\nAverage CO₂ by fuel type:")
print(avg_by_fuel.to_string(index=False))

# 6c. Average CO₂ emissions by state
avg_by_state = (
    merged_clean
    .groupby('state')['CO2EMISSIONS']
    .mean()
    .sort_values()
    .reset_index()
)
print("\nAverage CO₂ by state:")
print(avg_by_state.to_string(index=False))

Total registrations:    263578
Matched records:        9212
Unmatched records:      251480

Average CO₂ by fuel type:
         fuel  CO2EMISSIONS
       petrol    196.747879
hybrid_petrol    200.969723
       diesel    221.150000

Average CO₂ by state:
            state  CO2EMISSIONS
           Pahang    182.205882
            Sabah    182.220930
           Perlis    182.387755
         Kelantan    183.099099
            Kedah    185.548571
       Terengganu    186.218182
           Melaka    191.977011
  Negeri Sembilan    192.200000
          Sarawak    193.130719
      Rakan Niaga    195.616549
     Pulau Pinang    197.304659
         Selangor    197.992701
            Johor    199.213542
            Perak    199.695312
W.P. Kuala Lumpur    208.012796


7. **Calculate the average emissions for every combination of state and fuel type, printing the average CO₂ by fuel type only for Pahang AND turn the hierarchical index into a pivot table**

In [16]:
# 7. Hierarchical indexing & pivot

# 7a. Create MultiIndex of (state, fuel)
mi = (
    merged_clean
    .groupby(['state', 'fuel'])['CO2EMISSIONS']
    .mean()
)

# 7b. Example partial indexing for one state
print("\nPahang CO₂ by fuel type:")
print(mi.loc['Pahang'])

# 7c. Pivot to wide form: states as rows, fuel types as columns
emissions_table = mi.unstack()
print("\nPivot table (state × fuel) of average CO₂:")
print(emissions_table.head().to_string())


Pahang CO₂ by fuel type:
fuel
hybrid_petrol    188.166667
petrol           181.411111
Name: CO2EMISSIONS, dtype: float64

Pivot table (state × fuel) of average CO₂:
fuel             diesel  hybrid_petrol      petrol
state                                             
Johor               NaN     217.833333  197.520833
Kedah               NaN     177.750000  185.922156
Kelantan            NaN     184.000000  183.056604
Melaka              NaN     223.000000  189.262500
Negeri Sembilan     NaN     198.583333  190.984127


8. **Put two summary tables(states and fuels)side by side**

In [18]:
# 8. Combine state-level and fuel-level summaries side by side

state_stats = avg_by_state.set_index('state')
fuel_stats  = avg_by_fuel.set_index('fuel')

combined_stats = pd.concat(
    {'state_stats': state_stats, 'fuel_stats': fuel_stats},
    axis=1
)
print("\nCombined state & fuel statistics:")
print(combined_stats.head().to_string())


Combined state & fuel statistics:
          state_stats   fuel_stats
         CO2EMISSIONS CO2EMISSIONS
Pahang     182.205882          NaN
Sabah      182.220930          NaN
Perlis     182.387755          NaN
Kelantan   183.099099          NaN
Kedah      185.548571          NaN
